![](https://drive.google.com/uc?export=view&id=1-5X9OUkA-C2Ih1gOS9Jd7GmkTWUEpDg1)

**Asignatura:** *Machine Learning*

**Profesor:** Juan Bekios Calfa

**Fecha de entrega:** 23 de abril de 2026
---

# Laboratorio 01: Recolección de textos desde X, RSS y almacenamiento binario

**Asignatura:** Machine Learning  
**Profesor:** Dr. Juan Bekios Calfa  
**Ayudantes:** Sr. Víctor Jopia y Sr. Andrés Armijo  

## Propósito del notebook

Este cuaderno está pensado para ser usado en **Google Colab** por estudiantes de pregrado.  
La idea es avanzar paso a paso, con explicaciones simples, para:

1. Conectarse a **X** mediante su API.
2. Leer noticias desde el **RSS de Radio Cooperativa**.
3. Unificar los textos en un **DataFrame** con la estructura pedida en el laboratorio.
4. Aplicar un **preprocesamiento básico**.
5. Mostrar ejemplos iniciales de:
   - frecuencia de palabras,
   - TF-IDF,
   - clustering,
   - PCA.
6. Guardar los datos resultantes en un **archivo binario eficiente**.

## Estructura mínima esperada

El laboratorio pide una tabla con al menos estas columnas:

- `id`
- `fuente`
- `texto`

En este notebook agregaremos además columnas extra útiles, por ejemplo:

- `fecha`
- `url`
- `autor`
- `consulta`
- `texto_limpio`

## Idea pedagógica

El objetivo no es solo "hacer que funcione", sino **entender qué hace cada parte del pipeline**.
Por eso, antes de varias celdas encontrarás una explicación breve de:
- qué problema resuelve la celda,
- por qué se necesita,
- y qué salida produce.

## Mapa del notebook

Este cuaderno está organizado siguiendo la lógica general del laboratorio:

1. Instalación de librerías.
2. Importación de módulos.
3. Configuración de credenciales.
4. Recolección de datos desde X.
5. Recolección de datos desde Radio Cooperativa RSS.
6. Carga opcional de otras fuentes.
7. Construcción del DataFrame consolidado.
8. Guardado del corpus en formato binario.
9. Limpieza y preprocesamiento.
10. Frecuencia de palabras.
11. TF-IDF.
12. Clustering simple.
13. PCA para visualización.
14. Almacenamiento en MongoDB.
15. Verificación final.

Puedes ejecutar el notebook celda por celda en orden.


In [ ]:
# Instalación de librerías
# En Google Colab esta celda instala los paquetes necesarios para el laboratorio.
# Agregamos pyarrow para poder guardar archivos en formato Parquet.

!pip -q install pandas requests feedparser pymongo scikit-learn matplotlib nltk pyarrow


## Importación de librerías

Aquí importamos las herramientas principales del laboratorio.

- `pandas`: para construir tablas y guardar archivos.
- `requests`: para llamar la API de X.
- `feedparser`: para leer RSS.
- `pymongo`: para guardar datos en MongoDB.
- `re`: para limpieza con expresiones regulares.
- `Counter`: para contar palabras.
- `TfidfVectorizer`, `KMeans`, `PCA`: para representación, clustering y visualización.

## Nota sobre almacenamiento binario

Aunque `pickle` es una opción válida para guardar objetos de Python, para una gran cantidad de datos tabulares suele ser más conveniente usar **Parquet**, porque es un formato binario orientado a datos tabulares y permite compresión. En pandas, `DataFrame.to_parquet()` escribe el DataFrame en formato parquet binario y admite compresión; además requiere `pyarrow` o `fastparquet`. `DataFrame.to_pickle()` también existe y sirve bien cuando se busca compatibilidad rápida dentro de Python.


In [ ]:
import re
import json
import pandas as pd
import requests
import feedparser
import matplotlib.pyplot as plt

from collections import Counter
from datetime import datetime, timezone
from getpass import getpass

from pymongo import MongoClient
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

## Configuración general del notebook

En esta sección definimos parámetros simples que después reutilizaremos.

Puedes cambiar:
- la consulta para X,
- la cantidad de publicaciones a buscar,
- la cantidad de noticias RSS,
- el nombre de la base de datos en MongoDB,
- el nombre de la colección,
- y los nombres de los archivos binarios de salida.

### Recomendación práctica

Para este laboratorio dejaremos dos opciones:

- **Parquet**: recomendada para conjuntos grandes de datos tabulares.
- **Pickle**: útil como alternativa simple en Python.

También dejaremos configurados el nombre de la base de datos y la colección de MongoDB para que el notebook quede listo para usar cuando el estudiante disponga de una URI válida.


In [ ]:
# ------------------------------
# Parámetros generales
# ------------------------------

# Consulta general sobre universidades o tecnología
#X_QUERY = "universidad OR educación OR tecnología lang:es -is:retweet"

# Consulta sobre política mundial y nacional
X_QUERY = '((Chile OR chileno OR chilena OR gobierno OR congreso OR senado OR diputados OR presidencia OR presidente OR ministra OR ministerio OR "La Moneda" OR elecciones OR plebiscito OR constitución) (política OR politico OR politica OR gobierno OR legislativo OR ejecutivo)) OR ((internacional OR mundo OR geopolítica OR geopolitica OR diplomacia OR ONU OR OTAN OR "Estados Unidos" OR Rusia OR Ucrania OR China OR Europa OR Israel OR Gaza) (política OR politico OR politica OR elecciones OR gobierno OR conflicto OR guerra OR sanciones)) lang:es has:links -is:retweet -is:reply'

X_MAX_RESULTS = 500

COOPERATIVA_RSS_URL = "https://www.cooperativa.cl/noticias/site/tax/port/all/rss_3___1.xml"
COOPERATIVA_MAX_ITEMS = 25

# RSS opcional adicional
LATERCERA_RSS_URL = "https://www.latercera.com/arc/outboundfeeds/rss/?outputType=xml"
LATERCERA_MAX_ITEMS = 25

# Configuración de MongoDB
MONGO_DB_NAME = "LaboratorioML"
MONGO_COLLECTION_NAME = "corpus_textos"

# Archivos binarios de salida
PARQUET_OUTPUT_NAME = "corpus_consolidado_laboratorio_01.parquet"
PICKLE_OUTPUT_NAME = "corpus_consolidado_laboratorio_01.pkl"


## Credenciales seguras

### 1) Token de X
Para consultar datos desde X necesitas un **Bearer Token**.

### 2) URI de MongoDB
Para guardar datos necesitas una cadena de conexión de MongoDB, por ejemplo una URI de Atlas.

**Recomendación didáctica importante:**  
No escribas credenciales directamente en el notebook si luego lo compartirás.  
Es mejor ingresarlas con `getpass()` o usar secretos de Colab.

Si todavía no tienes credenciales, puedes dejar una de las dos vacía y seguir probando otras partes del laboratorio.

### Nota práctica para MongoDB
En este notebook ya dejamos definidos:
- el nombre de la base de datos (`MONGO_DB_NAME`),
- y el nombre de la colección (`MONGO_COLLECTION_NAME`).

Eso permite que el estudiante solo tenga que ingresar la URI cuando quiera probar el guardado real.


In [ ]:
# Puedes dejar estas variables vacías si aún no tienes credenciales
X_BEARER_TOKEN = getpass("Ingresa tu Bearer Token de X (o presiona Enter para omitir): ")
MONGODB_URI = getpass("Ingresa tu URI de MongoDB (o presiona Enter para omitir): ")

## Función para conectarse a X

Usaremos una función para mantener el código ordenado.

### ¿Qué hace esta función?
- Envía una consulta a la API de X.
- Recupera publicaciones recientes.
- Extrae campos básicos.
- Devuelve un `DataFrame`.

### ¿Por qué es útil?
Porque separa la lógica de conexión de la lógica del resto del laboratorio.  
Eso hace el código más modular y más fácil de entender.

In [ ]:
def buscar_posts_x(query, bearer_token, max_results=10):
    """
    Busca publicaciones recientes en X y devuelve un DataFrame.

    Parámetros:
    - query: texto de búsqueda en sintaxis de X.
    - bearer_token: token Bearer para autenticación.
    - max_results: cantidad de publicaciones a recuperar.

    Retorna:
    - DataFrame con columnas estandarizadas para el laboratorio.
    """
    if not bearer_token:
        print("No se ingresó Bearer Token. Se devolverá un DataFrame vacío.")
        return pd.DataFrame(columns=["id", "fuente", "texto", "fecha", "url", "autor", "consulta"])

    url = "https://api.x.com/2/tweets/search/recent"

    headers = {
        "Authorization": f"Bearer {bearer_token}"
    }

    params = {
        "query": query,
        "max_results": max(10, min(max_results, 100)),
        "tweet.fields": "created_at,author_id,lang",
        "expansions": "author_id",
        "user.fields": "username,name"
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)

    if response.status_code != 200:
        print("Error al consultar X:")
        print("Código:", response.status_code)
        print("Detalle:", response.text[:1000])
        return pd.DataFrame(columns=["id", "fuente", "texto", "fecha", "url", "autor", "consulta"])

    data = response.json()
    posts = data.get("data", [])
    users = data.get("includes", {}).get("users", [])

    # Mapa author_id -> username
    user_map = {u["id"]: u.get("username", "") for u in users}

    registros = []
    for post in posts:
        post_id = post.get("id", "")
        author_id = post.get("author_id", "")
        username = user_map.get(author_id, "")
        texto = post.get("text", "")

        registros.append({
            "id": post_id,
            "fuente": "x",
            "texto": texto,
            "fecha": post.get("created_at"),
            "url": f"https://x.com/{username}/status/{post_id}" if username and post_id else None,
            "autor": username if username else author_id,
            "consulta": query
        })

    return pd.DataFrame(registros)

## Ejemplo de uso: extracción desde X

Esta celda ejecuta la función anterior con una consulta simple.  
Si el token es válido, deberías obtener publicaciones recientes.

Si no tienes token, la celda no fallará: simplemente devolverá un DataFrame vacío.

In [ ]:
df_x = buscar_posts_x(
    query=X_QUERY,
    bearer_token=X_BEARER_TOKEN,
    max_results=X_MAX_RESULTS
)

print("Cantidad de publicaciones recuperadas desde X:", len(df_x))
df_x.head()

### Visualizar un poco más

In [ ]:
df_x.head(100)

### Guardar el df_x

In [ ]:
df_x.to_parquet("df_x.parquet", index=False)

### Cargar el df_x

In [ ]:
import pandas as pd

df_x = pd.read_parquet("df_x.parquet")

### Ver los textos

In [ ]:
for i, text in enumerate(df_x['texto']):
  print(f'#{i}: {text[:100]}', end='\n\n')

## Función para leer un RSS

Ahora construiremos una función genérica para RSS.

### ¿Qué hace?
- Lee un feed RSS.
- Toma título, resumen, enlace y fecha.
- Forma una tabla con el mismo formato general del laboratorio.

### ¿Por qué conviene hacer una función genérica?
Porque luego la misma función sirve para:
- Radio Cooperativa,
- La Tercera,
- u otros medios RSS.

In [ ]:
def leer_rss(url_rss, nombre_fuente, max_items=20):
    """
    Lee un RSS y devuelve un DataFrame con estructura estándar.
    """
    feed = feedparser.parse(url_rss)

    registros = []
    for i, entry in enumerate(feed.entries[:max_items], start=1):
        titulo = entry.get("title", "")
        resumen = entry.get("summary", "")
        enlace = entry.get("link", "")
        fecha = entry.get("published", entry.get("updated", None))

        texto = f"{titulo}. {resumen}".strip()

        registros.append({
            "id": f"{nombre_fuente}_{i}",
            "fuente": nombre_fuente,
            "texto": texto,
            "fecha": fecha,
            "url": enlace,
            "autor": None,
            "consulta": None
        })

    return pd.DataFrame(registros)

## Ejemplo claro y simple: Radio Cooperativa RSS

Esta es una de las fuentes pedidas en la guía del laboratorio.

La idea es que cada noticia se convierta en una fila del DataFrame.  
Para el campo `texto`, unimos:

- el **título**, y
- el **resumen**.

Eso hace que cada documento tenga más contenido útil para el análisis posterior.

In [ ]:
df_cooperativa = leer_rss(
    url_rss=COOPERATIVA_RSS_URL,
    nombre_fuente="cooperativa",
    max_items=COOPERATIVA_MAX_ITEMS
)

print("Cantidad de noticias recuperadas desde Radio Cooperativa:", len(df_cooperativa))
df_cooperativa.head()

## Ejemplo opcional: La Tercera RSS

Aunque usted pidió explícitamente Radio Cooperativa, dejo también este ejemplo porque el laboratorio menciona ambos medios RSS.  
La lógica es exactamente la misma.

In [ ]:
df_latercera = leer_rss(
    url_rss=LATERCERA_RSS_URL,
    nombre_fuente="latercera",
    max_items=LATERCERA_MAX_ITEMS
)

print("Cantidad de noticias recuperadas desde La Tercera:", len(df_latercera))
df_latercera.head()

## Fuente opcional: TikTok desde archivo CSV

En muchos cursos, el acceso directo a TikTok puede ser más complejo o depender de una muestra previa.  
Por eso, una alternativa muy práctica es cargar un archivo CSV preparado por el docente o por el grupo.

### Suposición simple
El archivo tiene al menos una columna llamada:
- `texto`
- `caption`
- `descripcion`

La función de abajo intenta adaptarse a esos nombres.

In [ ]:
def cargar_tiktok_desde_csv(ruta_csv):
    """
    Carga un CSV con textos de TikTok.
    Busca columnas típicas como: texto, caption o descripcion.
    """
    df = pd.read_csv(ruta_csv)

    posibles_columnas = ["texto", "caption", "descripcion", "description"]
    columna_texto = None

    for col in posibles_columnas:
        if col in df.columns:
            columna_texto = col
            break

    if columna_texto is None:
        raise ValueError(
            "No se encontró una columna válida de texto en el CSV. "
            "Use una columna llamada texto, caption, descripcion o description."
        )

    df_salida = pd.DataFrame({
        "id": [f"tiktok_{i+1}" for i in range(len(df))],
        "fuente": "tiktok",
        "texto": df[columna_texto].astype(str),
        "fecha": df[df.columns[0]].astype(str) if len(df.columns) > 0 else None,
        "url": None,
        "autor": None,
        "consulta": None
    })

    return df_salida

### Celda opcional para cargar TikTok

Descomenta esta celda si ya tienes un archivo CSV preparado en Colab.

In [ ]:
# EJEMPLO OPCIONAL:
# from google.colab import files
# uploaded = files.upload()
# nombre_archivo = list(uploaded.keys())[0]
# df_tiktok = cargar_tiktok_desde_csv(nombre_archivo)
# print("Cantidad de textos recuperados desde TikTok:", len(df_tiktok))
# df_tiktok.head()

# Mientras no se cargue un archivo real, dejamos un DataFrame vacío:
df_tiktok = pd.DataFrame(columns=["id", "fuente", "texto", "fecha", "url", "autor", "consulta"])

## Construcción del DataFrame consolidado

Esta es una parte central del laboratorio.

### ¿Qué queremos lograr?
Unir en una sola tabla los textos provenientes de:
- X,
- TikTok,
- Radio Cooperativa,
- La Tercera.

### ¿Por qué?
Porque después el análisis de frecuencia, TF-IDF, clustering y PCA se hace sobre un **corpus unificado**.

In [ ]:
# Unimos todas las fuentes disponibles
df_corpus = pd.concat(
    [df_x, df_tiktok, df_cooperativa, df_latercera],
    ignore_index=True
)

# Eliminamos filas sin texto real
df_corpus["texto"] = df_corpus["texto"].fillna("").astype(str)
df_corpus = df_corpus[df_corpus["texto"].str.strip() != ""].copy()

# Normalizamos el tipo de dato de algunas columnas
for col in ["id", "fuente", "fecha", "url", "autor", "consulta"]:
    if col in df_corpus.columns:
        df_corpus[col] = df_corpus[col].astype(str)

print("Cantidad total de documentos en el corpus:", len(df_corpus))
df_corpus.head(10)

## Verificación rápida del corpus

Antes de seguir, conviene revisar:
- cuántos textos tenemos por fuente,
- si hay valores vacíos,
- y si la estructura quedó bien.

Eso evita errores posteriores.

In [ ]:
print("Documentos por fuente:")
print(df_corpus["fuente"].value_counts(dropna=False))

print("\nColumnas del DataFrame:")
print(df_corpus.columns.tolist())

print("\nNúmero de valores nulos por columna:")
print(df_corpus.isnull().sum())

## Guardar el corpus consolidado en formato binario

Aquí cambiaremos el almacenamiento local del corpus.

### ¿Por qué no usar solo CSV?
El CSV es fácil de leer, pero cuando la cantidad de datos crece:
- ocupa más espacio,
- puede ser más lento,
- y pierde parte de la riqueza de tipos de datos.

### Formatos usados en este notebook

#### 1) Parquet
Será la opción principal.  
Es un formato binario pensado para datos tabulares y suele ser más adecuado para grandes volúmenes.

#### 2) Pickle
Lo dejaremos como alternativa.  
Es muy cómodo dentro de Python, aunque para intercambio y trabajo tabular grande normalmente **Parquet** suele ser mejor elección.

En pandas, `to_parquet()` permite elegir compresión y `to_pickle()` guarda el objeto serializado. citeturn0view0turn0view1


In [ ]:
# Guardado principal en Parquet
df_corpus.to_parquet(PARQUET_OUTPUT_NAME, index=False, compression="snappy")
print(f"Archivo Parquet guardado correctamente: {PARQUET_OUTPUT_NAME}")

# Guardado opcional en Pickle
df_corpus.to_pickle(PICKLE_OUTPUT_NAME)
print(f"Archivo Pickle guardado correctamente: {PICKLE_OUTPUT_NAME}")


## Verificación del archivo binario

Después de guardar, conviene comprobar que los datos pueden volver a cargarse correctamente.

Mostraremos dos ejemplos:
- lectura desde **Parquet**,
- lectura desde **Pickle**.

Normalmente, para este laboratorio bastará con trabajar desde Parquet.


In [ ]:
# Lectura de prueba desde Parquet
df_parquet = pd.read_parquet(PARQUET_OUTPUT_NAME)
print("Forma del DataFrame leído desde Parquet:", df_parquet.shape)
display(df_parquet.head())

# Lectura de prueba desde Pickle
df_pickle = pd.read_pickle(PICKLE_OUTPUT_NAME)
print("Forma del DataFrame leído desde Pickle:", df_pickle.shape)
display(df_pickle.head())


## Preprocesamiento básico de texto

La guía del laboratorio pide, como mínimo:

- minúsculas,
- eliminación de enlaces,
- eliminación de menciones y hashtags cuando corresponda,
- eliminación de números y signos de puntuación,
- eliminación de espacios repetidos,
- tokenización,
- eliminación de stop words.

Aquí implementamos una versión clara y didáctica de ese proceso.

In [ ]:
STOPWORDS_ES = set(stopwords.words("spanish"))

def limpiar_texto(texto):
    """
    Aplica limpieza básica a un texto en español.
    """
    texto = texto.lower()
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)         # enlaces
    # Eliminar también -->
    # menciones y hashtags
    # números
    # signos de puntuación y caracteres extra
    # espacios repetidos

    tokens = texto.split()
    tokens = [tok for tok in tokens if tok not in STOPWORDS_ES and len(tok) > 2]

    return " ".join(tokens)

df_corpus["texto_limpio"] = df_corpus["texto"].apply(limpiar_texto)

df_corpus[["fuente", "texto", "texto_limpio"]].head(10)

## Ejemplo didáctico de limpieza

Esta celda ayuda a que el estudiante vea **cómo cambia un texto** antes y después del preprocesamiento.

In [ ]:
if len(df_corpus) > 0:
    ejemplo = df_corpus.iloc[0]
    print("FUENTE:", ejemplo["fuente"])
    print("\nTEXTO ORIGINAL:\n")
    print(ejemplo["texto"])
    print("\nTEXTO LIMPIO:\n")
    print(ejemplo["texto_limpio"])
else:
    print("El corpus está vacío.")

## Frecuencia de palabras

La frecuencia de palabras permite detectar vocabulario dominante.

### Idea simple
Si ciertas palabras aparecen muchas veces en una fuente o en un conjunto de documentos,
pueden sugerir un tema dominante o una regularidad léxica.

In [ ]:
def obtener_frecuencias(textos):
    todas = " ".join(textos).split()
    return Counter(todas)

frecuencias_globales = obtener_frecuencias(df_corpus["texto_limpio"])

top_20_global = pd.DataFrame(
    frecuencias_globales.most_common(20),
    columns=["palabra", "frecuencia"]
)

top_20_global

## Frecuencia por fuente

Ahora repetimos la idea, pero separando por fuente.  
Eso ayuda a comparar si las palabras más comunes cambian entre X, TikTok y medios RSS.

In [ ]:
top_por_fuente = {}

for fuente in df_corpus["fuente"].unique():
    textos_fuente = df_corpus.loc[df_corpus["fuente"] == fuente, "texto_limpio"]
    frec = obtener_frecuencias(textos_fuente)
    top_por_fuente[fuente] = pd.DataFrame(
        frec.most_common(10),
        columns=["palabra", "frecuencia"]
    )

for fuente, tabla in top_por_fuente.items():
    print("=" * 60)
    print(f"Fuente: {fuente}")
    display(tabla)

## Gráfico simple de frecuencia global

Un gráfico sencillo ayuda a que los estudiantes visualicen de inmediato las palabras más repetidas.

In [ ]:
if not top_20_global.empty:
    plt.figure(figsize=(12, 5))
    plt.bar(top_20_global["palabra"], top_20_global["frecuencia"])
    plt.title("Top 20 palabras más frecuentes del corpus")
    plt.xlabel("Palabra")
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Representación TF-IDF

### ¿Qué significa TF-IDF?
Es una forma de convertir textos en números.

### Idea intuitiva
- Una palabra recibe más peso si aparece en un documento.
- Pero su peso baja si aparece en casi todos los documentos.

Eso ayuda a resaltar palabras más representativas.

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(df_corpus["texto_limpio"])

print("Forma de la matriz TF-IDF:", X_tfidf.shape)

## Palabras con mayor peso promedio en TF-IDF

Una forma simple de interpretar TF-IDF es calcular el peso promedio de cada término en todo el corpus.

In [ ]:
import numpy as np

terminos = vectorizer.get_feature_names_out()
pesos_promedio = np.asarray(X_tfidf.mean(axis=0)).ravel()

df_tfidf_global = pd.DataFrame({
    "termino": terminos,
    "peso_promedio": pesos_promedio
}).sort_values("peso_promedio", ascending=False)

df_tfidf_global.head(20)

## Clustering simple con K-Means

Aquí agrupamos documentos por similitud léxica.

### Idea simple
El algoritmo intenta encontrar grupos de documentos que "se parezcan" en su representación TF-IDF.

### Nota didáctica
Elegimos `k=4` solo como ejemplo porque el laboratorio trabaja con cuatro fuentes principales.  
Eso no significa que 4 sea siempre el mejor número de clusters.

In [ ]:
k = 4 if len(df_corpus) >= 4 else max(1, len(df_corpus))

if k >= 1 and len(df_corpus) > 0:
    modelo_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df_corpus["cluster"] = modelo_kmeans.fit_predict(X_tfidf)
else:
    df_corpus["cluster"] = 0

df_corpus[["fuente", "texto_limpio", "cluster"]].head(10)

## Términos más representativos de cada cluster

Una manera sencilla de interpretar cada cluster es mirar qué palabras tienen mayor peso en el centroide del grupo.

In [ ]:
def top_terminos_por_cluster(modelo, vectorizer, top_n=10):
    terminos = vectorizer.get_feature_names_out()
    resultados = {}

    for i, centroide in enumerate(modelo.cluster_centers_):
        indices = centroide.argsort()[::-1][:top_n]
        resultados[i] = [terminos[idx] for idx in indices]

    return resultados

if len(df_corpus) > 0 and "modelo_kmeans" in globals():
    top_clusters = top_terminos_por_cluster(modelo_kmeans, vectorizer, top_n=10)
    for cluster_id, terminos in top_clusters.items():
        print(f"Cluster {cluster_id}: {terminos}")

## Reducción de dimensionalidad con PCA

La matriz TF-IDF puede tener cientos o miles de dimensiones.  
PCA permite proyectarla a solo **2 dimensiones** para visualizar mejor los documentos.

### Importante
PCA no "entiende" el significado del texto.  
Solo busca resumir la variación numérica de los datos.

In [ ]:
if len(df_corpus) >= 2:
    X_dense = X_tfidf.toarray()
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_dense)

    df_corpus["pca_1"] = X_pca[:, 0]
    df_corpus["pca_2"] = X_pca[:, 1]
else:
    df_corpus["pca_1"] = 0.0
    df_corpus["pca_2"] = 0.0

df_corpus[["fuente", "cluster", "pca_1", "pca_2"]].head()

## Gráfico PCA coloreado por fuente

In [ ]:
if len(df_corpus) > 0:
    plt.figure(figsize=(8, 6))

    for fuente in df_corpus["fuente"].unique():
        subset = df_corpus[df_corpus["fuente"] == fuente]
        plt.scatter(subset["pca_1"], subset["pca_2"], label=fuente, alpha=0.7)

    plt.title("PCA del corpus coloreado por fuente")
    plt.xlabel("Componente principal 1")
    plt.ylabel("Componente principal 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Gráfico PCA coloreado por cluster

In [ ]:
if len(df_corpus) > 0:
    plt.figure(figsize=(8, 6))

    for cluster_id in sorted(df_corpus["cluster"].unique()):
        subset = df_corpus[df_corpus["cluster"] == cluster_id]
        plt.scatter(subset["pca_1"], subset["pca_2"], label=f"Cluster {cluster_id}", alpha=0.7)

    plt.title("PCA del corpus coloreado por cluster")
    plt.xlabel("Componente principal 1")
    plt.ylabel("Componente principal 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Guardar datos en MongoDB

Ahora llevamos el corpus a MongoDB.

### Estrategia didáctica
En vez de insertar filas "a ciegas", usaremos `upsert`.  
Eso significa:

- si el documento ya existe, se actualiza;
- si no existe, se crea.

Esto evita duplicados innecesarios cuando el notebook se ejecuta varias veces.

### Configuración usada en este notebook
El guardado utilizará:
- `MONGO_DB_NAME` como nombre de la base de datos,
- `MONGO_COLLECTION_NAME` como nombre de la colección.

Así, el estudiante puede cambiar esos valores desde la sección de parámetros sin tocar la lógica de la función.


In [ ]:
def guardar_en_mongodb(df, mongo_uri, db_name, collection_name):
    """
    Guarda un DataFrame en MongoDB usando upsert por _id.
    """
    if not mongo_uri:
        print("No se ingresó URI de MongoDB. Se omite el guardado.")
        return

    if df.empty:
        print("El DataFrame está vacío. No hay datos para guardar.")
        return

    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]

    fecha_carga = datetime.now(timezone.utc).isoformat()

    for _, row in df.iterrows():
        documento = row.to_dict()

        # Creamos una llave única estable para evitar duplicados
        documento["_id"] = f"{documento['fuente']}::{documento['id']}"
        documento["fecha_carga_utc"] = fecha_carga

        collection.update_one(
            {"_id": documento["_id"]},
            {"$set": documento},
            upsert=True
        )

    print(f"Datos guardados/actualizados en MongoDB: {db_name}.{collection_name}")

## Ejecutar guardado en MongoDB

Si la URI es válida y el corpus contiene datos, esta celda realizará el guardado o actualización de documentos en la colección configurada.


In [ ]:
guardar_en_mongodb(
    df=df_corpus,
    mongo_uri=MONGODB_URI,
    db_name=MONGO_DB_NAME,
    collection_name=MONGO_COLLECTION_NAME
)

## Verificación simple desde MongoDB

Esta celda comprueba:
- si la conexión funciona,
- cuántos documentos hay,
- y muestra algunos registros guardados.

In [ ]:
def verificar_mongodb(mongo_uri, db_name, collection_name, limite=5):
    if not mongo_uri:
        print("No se ingresó URI de MongoDB. Se omite la verificación.")
        return

    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]

    total = collection.count_documents({})
    print("Total de documentos en la colección:", total)

    docs = list(collection.find({}, {"_id": 1, "fuente": 1, "texto": 1}).limit(limite))
    for doc in docs:
        print("-" * 80)
        print(json.dumps(doc, ensure_ascii=False, indent=2, default=str))

verificar_mongodb(
    mongo_uri=MONGODB_URI,
    db_name=MONGO_DB_NAME,
    collection_name=MONGO_COLLECTION_NAME
)

## Preguntas guía para interpretar los resultados

Después de ejecutar el notebook, los estudiantes pueden discutir:

1. ¿Qué palabras aparecen con mayor frecuencia en todo el corpus?
2. ¿Qué diferencias léxicas se observan entre X y los medios RSS?
3. ¿Qué palabras parecen caracterizar cada cluster?
4. ¿La visualización PCA separa relativamente bien las fuentes?
5. ¿Qué evidencia exploratoria existe para hablar de patrones de discurso?
6. ¿Qué limitaciones tiene este enfoque?

## Limitaciones importantes

Este análisis es útil como primer acercamiento, pero tiene límites:

- No capta ironía ni contexto profundo.
- No distingue bien polisemia.
- La frecuencia no equivale automáticamente a "discurso".
- Los resultados dependen mucho de la muestra recolectada.
- La calidad del preprocesamiento influye bastante en el resultado final.

## Cierre

Con este notebook ya tienes un punto de partida completo para:
- recolectar textos,
- unificarlos,
- limpiarlos,
- representarlos numéricamente,
- agruparlos,
- visualizarlos,
- guardarlos en formato binario,
- y guardarlos en MongoDB.

La idea es que luego cada grupo pueda extenderlo, modularizarlo mejor y adaptarlo a su propio tema de análisis.